# 統計分析 中級編（JavaScript）

このノートブックでは、JavaScript を使って **統計分析の考え方と手順** を、たくさんの例題を通して学びます。
記述統計から確率分布、推定、仮説検定、相関・回帰、時系列の基礎まで扱います。

外部ライブラリは使わず、必要な関数（平均・分散から t 分布・カイ二乗分布の計算まで）を **すべて自分で定義** します。
そのため「統計の計算の中身」がそのまま見えるのがこのノートブックの特長です。

## このノートブックの使い方

- コードセルを上から順番に **Shift + Enter** で実行してください。前半の「準備」のセルで定義した関数を、後の例題で使います。
- セルの **最後の行** にセミコロン（`;`）を付けずに式を書くと、その値が表示されます。グラフはこの仕組みで表示しています（`histogram(...)` などの呼び出しを最後の行に書く）。
- `console.log()` の出力はセルの下に表示されます。
- **注意**: JupyterLite の JavaScript カーネルには、**Run All Cells** で全セルを連続実行すると出力が次のセルにずれて表示されることがある既知の問題があります。**1 セルずつ** 実行してください。

## 前提知識

- JavaScript の基本文法（`javascript-tutorial.ipynb` の内容: 配列、関数、オブジェクト、アロー関数、`map` / `filter` / `reduce`）
- 高校レベルの数学（平均、割合、平方根、指数・対数）

## 目次

1. 準備（統計関数・乱数・グラフの定義）
2. 記述統計（例題 1〜9）
3. 確率分布とシミュレーション（例題 10〜15）
4. 推定（例題 16〜18）
5. 仮説検定（例題 19〜31）
6. 相関と回帰（例題 32〜37）
7. 時系列の基礎（例題 38〜39）
8. 練習問題

## 1. 準備

まず、このノートブック全体で使う関数を定義します。ここは読み飛ばしても構いませんが、**必ず実行** してください。

### 1.1 基本的な統計量

| 関数 | 内容 |
|---|---|
| `mean(a)` / `median(a)` / `mode(a)` | 平均・中央値・最頻値 |
| `variance(a, ddof)` / `sd(a, ddof)` | 分散・標準偏差（`ddof = 1` で標本分散、`0` で母分散） |
| `quantile(a, p)` / `iqr(a)` | 分位数（`p = 0.25` で第 1 四分位数）・四分位範囲 |
| `zscores(a)` | 標準化（z スコア） |
| `skewness(a)` / `kurtosis(a)` | 歪度・尖度 |
| `cov(x, y)` / `corr(x, y)` / `spearman(x, y)` | 共分散・ピアソン相関・スピアマン順位相関 |
| `linearRegression(x, y)` | 単回帰（傾き・切片・決定係数・残差） |

In [ ]:
// ---- 基本 ----
function round(x, digits = 2) {
  const p = 10 ** digits;
  return Math.round(x * p) / p;
}

function sum(a) {
  return a.reduce((s, v) => s + v, 0);
}

function mean(a) {
  return sum(a) / a.length;
}

function median(a) {
  const s = [...a].sort((x, y) => x - y);
  const m = Math.floor(s.length / 2);
  return s.length % 2 === 1 ? s[m] : (s[m - 1] + s[m]) / 2;
}

function mode(a) {
  const counts = new Map();
  for (const v of a) counts.set(v, (counts.get(v) || 0) + 1);
  let best = null, bestCount = 0;
  for (const [v, c] of counts) {
    if (c > bestCount) { best = v; bestCount = c; }
  }
  return best;
}

// ddof = 1: 標本分散（n - 1 で割る）、ddof = 0: 母分散（n で割る）
function variance(a, ddof = 1) {
  const m = mean(a);
  return sum(a.map((v) => (v - m) ** 2)) / (a.length - ddof);
}

function sd(a, ddof = 1) {
  return Math.sqrt(variance(a, ddof));
}

// 分位数（線形補間。numpy / pandas の既定と同じ方法）
function quantile(a, p) {
  const s = [...a].sort((x, y) => x - y);
  const idx = (s.length - 1) * p;
  const lo = Math.floor(idx), hi = Math.ceil(idx);
  return s[lo] + (s[hi] - s[lo]) * (idx - lo);
}

function iqr(a) {
  return quantile(a, 0.75) - quantile(a, 0.25);
}

function zscores(a) {
  const m = mean(a), s = sd(a);
  return a.map((v) => (v - m) / s);
}

function skewness(a) {
  const m = mean(a), s = sd(a, 0);
  return mean(a.map((v) => ((v - m) / s) ** 3));
}

function kurtosis(a) {
  const m = mean(a), s = sd(a, 0);
  return mean(a.map((v) => ((v - m) / s) ** 4)) - 3;   // 正規分布で 0 になる定義
}

// ---- 2 変数 ----
function cov(x, y) {
  const mx = mean(x), my = mean(y);
  return sum(x.map((v, i) => (v - mx) * (y[i] - my))) / (x.length - 1);
}

function corr(x, y) {
  return cov(x, y) / (sd(x) * sd(y));
}

// 順位（同じ値には平均順位を与える）
function rank(a) {
  const idx = a.map((v, i) => [v, i]).sort((p, q) => p[0] - q[0]);
  const r = new Array(a.length);
  let i = 0;
  while (i < idx.length) {
    let j = i;
    while (j + 1 < idx.length && idx[j + 1][0] === idx[i][0]) j++;
    const avg = (i + j) / 2 + 1;
    for (let k = i; k <= j; k++) r[idx[k][1]] = avg;
    i = j + 1;
  }
  return r;
}

function spearman(x, y) {
  return corr(rank(x), rank(y));
}

function linearRegression(x, y) {
  const slope = cov(x, y) / variance(x);
  const intercept = mean(y) - slope * mean(x);
  const predicted = x.map((v) => intercept + slope * v);
  const residuals = y.map((v, i) => v - predicted[i]);
  const ssRes = sum(residuals.map((r) => r ** 2));
  const ssTot = sum(y.map((v) => (v - mean(y)) ** 2));
  return {
    slope, intercept,
    r2: 1 - ssRes / ssTot,
    residuals,
    predict: (v) => intercept + slope * v,
  };
}

// 要約統計量をまとめて返す
function summary(a) {
  return {
    n: a.length, mean: mean(a), sd: sd(a), min: Math.min(...a),
    q1: quantile(a, 0.25), median: median(a), q3: quantile(a, 0.75), max: Math.max(...a),
  };
}

console.log("統計関数を定義しました");

### 1.2 乱数（シード付き）

`Math.random()` は毎回違う結果になるので、**シード（種）を指定できる乱数** を自分で用意します。同じシードなら同じ結果が再現できます。

- `setSeed(n)`: シードを設定する
- `rand()`: 0 以上 1 未満の乱数
- `randNormal(mean, sd)`: 正規分布に従う乱数（Box-Muller 法）
- `randInt(min, max)`: min 以上 max 以下の整数
- `shuffle(a)` / `resample(a)`: シャッフル / 復元抽出（ブートストラップ用）

In [ ]:
// mulberry32 と呼ばれる小さな乱数生成アルゴリズム
function makeRandom(seed) {
  let a = seed >>> 0;
  return function () {
    a = (a + 0x6D2B79F5) >>> 0;
    let t = a;
    t = Math.imul(t ^ (t >>> 15), t | 1);
    t ^= t + Math.imul(t ^ (t >>> 7), t | 61);
    return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
  };
}

const rng = { next: makeRandom(42) };   // オブジェクトにしておくと、後のセルからシードを変えられる

function setSeed(seed) {
  rng.next = makeRandom(seed);
}

function rand() {
  return rng.next();
}

function randNormal(mu = 0, sigma = 1) {
  const u = 1 - rand(), v = rand();   // u が 0 にならないように
  return mu + sigma * Math.sqrt(-2 * Math.log(u)) * Math.cos(2 * Math.PI * v);
}

function randInt(min, max) {
  return min + Math.floor(rand() * (max - min + 1));
}

function shuffle(a) {
  const s = [...a];
  for (let i = s.length - 1; i > 0; i--) {
    const j = Math.floor(rand() * (i + 1));
    [s[i], s[j]] = [s[j], s[i]];
  }
  return s;
}

function resample(a) {
  return a.map(() => a[Math.floor(rand() * a.length)]);
}

// n 個の乱数を配列で作る
function generate(n, fn) {
  return Array.from({ length: n }, fn);
}

setSeed(1);
console.log("乱数の例:", generate(5, () => round(rand(), 3)));

### 1.3 確率分布の関数

検定で p 値を計算するには、t 分布やカイ二乗分布の **累積分布関数（CDF）** が必要です。数値計算の定番の方法（不完全ベータ関数・不完全ガンマ関数の連分数展開）で実装します。
中身を理解する必要はありません。「こういう関数がある」ことだけ押さえておけば十分です。

| 関数 | 内容 |
|---|---|
| `normalPdf(x, mu, sigma)` / `normalCdf(x, mu, sigma)` | 正規分布の確率密度 / 累積確率 |
| `normalQuantile(p)` | 標準正規分布の分位点（`normalQuantile(0.975)` ≒ 1.96） |
| `tCdf(t, df)` / `tQuantile(p, df)` | t 分布の累積確率 / 分位点 |
| `chi2Cdf(x, df)` | カイ二乗分布の累積確率 |
| `fCdf(f, df1, df2)` | F 分布の累積確率 |
| `binomialPmf(k, n, p)` / `poissonPmf(k, lambda)` | 二項分布 / ポアソン分布の確率 |

In [ ]:
// ガンマ関数の対数（Lanczos 近似）
function lnGamma(x) {
  const cof = [76.18009172947146, -86.50532032941677, 24.01409824083091,
               -1.231739572450155, 0.1208650973866179e-2, -0.5395239384953e-5];
  let y = x, tmp = x + 5.5;
  tmp -= (x + 0.5) * Math.log(tmp);
  let ser = 1.000000000190015;
  for (let j = 0; j < 6; j++) ser += cof[j] / ++y;
  return -tmp + Math.log(2.5066282746310005 * ser / x);
}

// 正則化不完全ガンマ関数 P(a, x)
function incompleteGamma(a, x) {
  if (x <= 0) return 0;
  if (x < a + 1) {                       // 級数展開
    let ap = a, s = 1 / a, del = s;
    for (let n = 1; n <= 500; n++) {
      ap += 1; del *= x / ap; s += del;
      if (Math.abs(del) < Math.abs(s) * 3e-14) break;
    }
    return s * Math.exp(-x + a * Math.log(x) - lnGamma(a));
  }
  let b = x + 1 - a, c = 1 / 1e-300, d = 1 / b, h = d;   // 連分数展開
  for (let i = 1; i <= 500; i++) {
    const an = -i * (i - a);
    b += 2;
    d = an * d + b; if (Math.abs(d) < 1e-300) d = 1e-300;
    c = b + an / c; if (Math.abs(c) < 1e-300) c = 1e-300;
    d = 1 / d;
    const del = d * c;
    h *= del;
    if (Math.abs(del - 1) < 3e-14) break;
  }
  return 1 - Math.exp(-x + a * Math.log(x) - lnGamma(a)) * h;
}

// 不完全ベータ関数の連分数部分
function betacf(a, b, x) {
  const qab = a + b, qap = a + 1, qam = a - 1;
  let c = 1, d = 1 - qab * x / qap;
  if (Math.abs(d) < 1e-300) d = 1e-300;
  d = 1 / d;
  let h = d;
  for (let m = 1; m <= 300; m++) {
    const m2 = 2 * m;
    let aa = m * (b - m) * x / ((qam + m2) * (a + m2));
    d = 1 + aa * d; if (Math.abs(d) < 1e-300) d = 1e-300;
    c = 1 + aa / c; if (Math.abs(c) < 1e-300) c = 1e-300;
    d = 1 / d; h *= d * c;
    aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2));
    d = 1 + aa * d; if (Math.abs(d) < 1e-300) d = 1e-300;
    c = 1 + aa / c; if (Math.abs(c) < 1e-300) c = 1e-300;
    d = 1 / d;
    const del = d * c;
    h *= del;
    if (Math.abs(del - 1) < 3e-14) break;
  }
  return h;
}

// 正則化不完全ベータ関数 I_x(a, b)
function incompleteBeta(x, a, b) {
  if (x <= 0) return 0;
  if (x >= 1) return 1;
  const bt = Math.exp(lnGamma(a + b) - lnGamma(a) - lnGamma(b) + a * Math.log(x) + b * Math.log(1 - x));
  if (x < (a + 1) / (a + b + 2)) return bt * betacf(a, b, x) / a;
  return 1 - bt * betacf(b, a, 1 - x) / b;
}

// ---- 正規分布 ----
function erf(x) {
  const p = incompleteGamma(0.5, x * x);
  return x >= 0 ? p : -p;
}

function normalPdf(x, mu = 0, sigma = 1) {
  const z = (x - mu) / sigma;
  return Math.exp(-0.5 * z * z) / (sigma * Math.sqrt(2 * Math.PI));
}

function normalCdf(x, mu = 0, sigma = 1) {
  return 0.5 * (1 + erf((x - mu) / (sigma * Math.SQRT2)));
}

// 二分法で分位点を求める（cdf(x) = p となる x）
function invert(cdf, p, lo, hi) {
  for (let i = 0; i < 200; i++) {
    const mid = (lo + hi) / 2;
    if (cdf(mid) < p) lo = mid; else hi = mid;
  }
  return (lo + hi) / 2;
}

function normalQuantile(p) {
  return invert(normalCdf, p, -12, 12);
}

// ---- t 分布・カイ二乗分布・F 分布 ----
function tCdf(t, df) {
  const x = df / (df + t * t);
  const p = 0.5 * incompleteBeta(x, df / 2, 0.5);
  return t >= 0 ? 1 - p : p;
}

function tQuantile(p, df) {
  return invert((t) => tCdf(t, df), p, -200, 200);
}

function chi2Cdf(x, df) {
  return incompleteGamma(df / 2, x / 2);
}

function fCdf(f, df1, df2) {
  if (f <= 0) return 0;
  return incompleteBeta(df1 * f / (df1 * f + df2), df1 / 2, df2 / 2);
}

// ---- 離散分布 ----
function lnChoose(n, k) {
  return lnGamma(n + 1) - lnGamma(k + 1) - lnGamma(n - k + 1);
}

function binomialPmf(k, n, p) {
  if (k < 0 || k > n) return 0;
  return Math.exp(lnChoose(n, k) + k * Math.log(p) + (n - k) * Math.log(1 - p));
}

function poissonPmf(k, lambda) {
  return Math.exp(k * Math.log(lambda) - lambda - lnGamma(k + 1));
}

// 動作確認（よく知られた値と比べる）
console.log("normalCdf(1.96)      =", round(normalCdf(1.96), 4), "（≒ 0.975）");
console.log("normalQuantile(0.975)=", round(normalQuantile(0.975), 4), "（≒ 1.96）");
console.log("tQuantile(0.975, 10) =", round(tQuantile(0.975, 10), 4), "（≒ 2.228）");
console.log("chi2Cdf(3.841, 1)    =", round(chi2Cdf(3.841, 1), 4), "（≒ 0.95）");
console.log("fCdf(4.26, 2, 9)     =", round(fCdf(4.26, 2, 9), 4), "（≒ 0.95）");

### 1.4 グラフ（SVG）

グラフはライブラリを使わず、**SVG の文字列を組み立てて** 表示します。セルの最後の行にグラフ関数の呼び出しを書く（セミコロンなし）と描画されます。

| 関数 | 内容 |
|---|---|
| `histogram(data, {bins, title})` | ヒストグラム |
| `scatter(x, y, {title, xLabel, yLabel, line})` | 散布図（`line: {slope, intercept}` で直線を重ねる） |
| `barChart(labels, values, {title})` | 棒グラフ |
| `lineChart([{name, values}], {xValues, title})` | 折れ線グラフ |
| `boxPlot({グループ名: データ}, {title})` | 箱ひげ図 |
| `htmlTable(headers, rows)` | 表 |

In [ ]:
// 目盛りの「きりのよい値」を作る
function niceTicks(min, max, count = 5) {
  const span = max - min || 1;
  const step0 = span / count;
  const mag = 10 ** Math.floor(Math.log10(step0));
  const norm = step0 / mag;
  const step = (norm < 1.5 ? 1 : norm < 3 ? 2 : norm < 7 ? 5 : 10) * mag;
  const out = [];
  for (let v = Math.ceil(min / step) * step; v <= max + step * 1e-6; v += step) out.push(+v.toFixed(10));
  return out;
}

// 軸・目盛り・タイトル付きの枠を描き、中身は body(sx, sy) で描く
function svgFrame(opt, body) {
  const { width = 480, height = 300, title = "", xLabel = "", yLabel = "",
          xMin, xMax, yMin, yMax, xTicks, xTickLabels } = opt;
  const m = { top: 34, right: 16, bottom: 48, left: 58 };
  const w = width - m.left - m.right, h = height - m.top - m.bottom;
  const sx = (x) => m.left + ((x - xMin) / (xMax - xMin)) * w;
  const sy = (y) => m.top + h - ((y - yMin) / (yMax - yMin)) * h;
  let s = `<svg xmlns="http://www.w3.org/2000/svg" width="${width}" height="${height}" ` +
          `style="font-family:sans-serif;font-size:12px;background:#fff">`;
  for (const v of niceTicks(yMin, yMax)) {
    const y = sy(v);
    s += `<line x1="${m.left}" x2="${m.left + w}" y1="${y}" y2="${y}" stroke="#eee"/>`;
    s += `<text x="${m.left - 6}" y="${y + 4}" text-anchor="end">${v}</text>`;
  }
  const xt = xTicks || niceTicks(xMin, xMax);
  xt.forEach((v, i) => {
    const x = sx(v);
    const label = xTickLabels ? xTickLabels[i] : v;
    s += `<line x1="${x}" x2="${x}" y1="${m.top + h}" y2="${m.top + h + 4}" stroke="#333"/>`;
    s += `<text x="${x}" y="${m.top + h + 18}" text-anchor="middle">${label}</text>`;
  });
  s += `<line x1="${m.left}" x2="${m.left + w}" y1="${m.top + h}" y2="${m.top + h}" stroke="#333"/>`;
  s += `<line x1="${m.left}" x2="${m.left}" y1="${m.top}" y2="${m.top + h}" stroke="#333"/>`;
  if (title) s += `<text x="${width / 2}" y="20" text-anchor="middle" font-size="14" font-weight="bold">${title}</text>`;
  if (xLabel) s += `<text x="${m.left + w / 2}" y="${height - 8}" text-anchor="middle">${xLabel}</text>`;
  if (yLabel) s += `<text transform="translate(14,${m.top + h / 2}) rotate(-90)" text-anchor="middle">${yLabel}</text>`;
  s += body(sx, sy, { m, w, h });
  return s + "</svg>";
}

const COLORS = ["#4e79a7", "#e15759", "#59a14f", "#f28e2b", "#76b7b2", "#b07aa1"];

function histogram(data, { bins = 10, title = "", xLabel = "値", yLabel = "度数", color = COLORS[0] } = {}) {
  const lo = Math.min(...data), hi = Math.max(...data);
  const step = (hi - lo) / bins || 1;
  const counts = new Array(bins).fill(0);
  for (const v of data) {
    let i = Math.floor((v - lo) / step);
    if (i >= bins) i = bins - 1;
    counts[i]++;
  }
  return svgFrame({ title, xLabel, yLabel, xMin: lo, xMax: hi, yMin: 0, yMax: Math.max(...counts) * 1.1 },
    (sx, sy) => counts.map((c, i) => {
      const x0 = sx(lo + i * step), x1 = sx(lo + (i + 1) * step);
      return `<rect x="${x0}" y="${sy(c)}" width="${Math.max(x1 - x0 - 1, 1)}" height="${sy(0) - sy(c)}" fill="${color}"/>`;
    }).join(""));
}

function scatter(x, y, { title = "", xLabel = "x", yLabel = "y", line = null, color = COLORS[0] } = {}) {
  const pad = (a) => { const lo = Math.min(...a), hi = Math.max(...a), p = (hi - lo || 1) * 0.08; return [lo - p, hi + p]; };
  const [xMin, xMax] = pad(x), [yMin, yMax] = pad(y);
  return svgFrame({ title, xLabel, yLabel, xMin, xMax, yMin, yMax }, (sx, sy) => {
    let s = x.map((v, i) => `<circle cx="${sx(v)}" cy="${sy(y[i])}" r="4" fill="${color}" fill-opacity="0.7"/>`).join("");
    if (line) {
      s += `<line x1="${sx(xMin)}" y1="${sy(line.intercept + line.slope * xMin)}" ` +
           `x2="${sx(xMax)}" y2="${sy(line.intercept + line.slope * xMax)}" stroke="#e15759" stroke-width="2"/>`;
    }
    return s;
  });
}

function barChart(labels, values, { title = "", xLabel = "", yLabel = "", color = COLORS[0] } = {}) {
  const n = labels.length;
  const yMax = Math.max(...values, 0) * 1.1 || 1;
  const yMin = Math.min(...values, 0);
  return svgFrame({ title, xLabel, yLabel, xMin: 0, xMax: n, yMin, yMax,
                    xTicks: labels.map((_, i) => i + 0.5), xTickLabels: labels },
    (sx, sy) => values.map((v, i) =>
      `<rect x="${sx(i + 0.15)}" y="${sy(Math.max(v, 0))}" width="${sx(i + 0.85) - sx(i + 0.15)}" ` +
      `height="${Math.abs(sy(v) - sy(0))}" fill="${color}"/>`).join(""));
}

function lineChart(series, { title = "", xLabel = "", yLabel = "", xValues = null } = {}) {
  const n = Math.max(...series.map((s) => s.values.length));
  const xs = xValues || [...Array(n).keys()];
  const all = series.flatMap((s) => s.values);
  const yMin = Math.min(...all), yMax = Math.max(...all), p = (yMax - yMin || 1) * 0.08;
  return svgFrame({ title, xLabel, yLabel, xMin: xs[0], xMax: xs[xs.length - 1], yMin: yMin - p, yMax: yMax + p },
    (sx, sy, g) => {
      let s = series.map((ser, k) =>
        `<polyline fill="none" stroke="${ser.color || COLORS[k % COLORS.length]}" stroke-width="2" ` +
        `points="${ser.values.map((v, i) => `${sx(xs[i])},${sy(v)}`).join(" ")}"/>`).join("");
      series.forEach((ser, k) => {
        s += `<text x="${g.m.left + g.w - 4}" y="${g.m.top + 14 + k * 16}" text-anchor="end" ` +
             `fill="${ser.color || COLORS[k % COLORS.length]}">${ser.name}</text>`;
      });
      return s;
    });
}

function boxPlot(groups, { title = "", yLabel = "値" } = {}) {
  const names = Object.keys(groups);
  const all = names.flatMap((k) => groups[k]);
  const yMin = Math.min(...all), yMax = Math.max(...all), p = (yMax - yMin || 1) * 0.08;
  return svgFrame({ title, yLabel, xMin: 0, xMax: names.length, yMin: yMin - p, yMax: yMax + p,
                    xTicks: names.map((_, i) => i + 0.5), xTickLabels: names },
    (sx, sy) => names.map((k, i) => {
      const v = groups[k];
      const q1 = quantile(v, 0.25), q2 = median(v), q3 = quantile(v, 0.75), r = q3 - q1;
      const inside = v.filter((d) => d >= q1 - 1.5 * r && d <= q3 + 1.5 * r);
      const lo = Math.min(...inside), hi = Math.max(...inside);
      const cx = sx(i + 0.5), bw = sx(0.6) - sx(0.3);
      let s = `<line x1="${cx}" x2="${cx}" y1="${sy(lo)}" y2="${sy(hi)}" stroke="#333"/>`;
      s += `<rect x="${cx - bw / 2}" y="${sy(q3)}" width="${bw}" height="${sy(q1) - sy(q3)}" fill="#a0cbe8" stroke="#333"/>`;
      s += `<line x1="${cx - bw / 2}" x2="${cx + bw / 2}" y1="${sy(q2)}" y2="${sy(q2)}" stroke="#e15759" stroke-width="2"/>`;
      s += v.filter((d) => d < q1 - 1.5 * r || d > q3 + 1.5 * r)
            .map((d) => `<circle cx="${cx}" cy="${sy(d)}" r="3" fill="none" stroke="#333"/>`).join("");
      return s;
    }).join(""));
}

function htmlTable(headers, rows) {
  const cell = (c) => typeof c === "number" ? round(c, 3) : c;
  const th = headers.map((h) => `<th style="border:1px solid #ccc;padding:4px 10px;background:#f3f3f3">${h}</th>`).join("");
  const trs = rows.map((r) => `<tr>${r.map((c) =>
    `<td style="border:1px solid #ccc;padding:4px 10px;text-align:right">${cell(c)}</td>`).join("")}</tr>`).join("");
  return `<table style="border-collapse:collapse;font-size:13px"><tr>${th}</tr>${trs}</table>`;
}

// 要約統計量を表にする
function summaryTable(named) {
  const keys = ["n", "mean", "sd", "min", "q1", "median", "q3", "max"];
  return htmlTable(["", ...keys], Object.entries(named).map(([k, a]) => [k, ...keys.map((s) => summary(a)[s])]));
}

console.log("グラフ関数を定義しました");

In [ ]:
// 準備の動作確認: 正規乱数のヒストグラム
setSeed(7);
const demo = generate(500, () => randNormal(50, 10));
console.log("平均:", round(mean(demo)), " 標準偏差:", round(sd(demo)));
histogram(demo, { bins: 20, title: "準備の確認: 正規乱数 500 個（平均 50, SD 10）" })

## 2. 記述統計

データの特徴を数値とグラフで要約する方法です。分析の最初に必ず行います。

### 例題 1: 代表値（平均・中央値・最頻値）

30 人のテストの点数があります。代表値を求め、どの値が「代表」としてふさわしいか考えましょう。

In [ ]:
const scores = [72, 85, 90, 66, 78, 95, 58, 81, 77, 88, 69, 74, 92, 83, 60,
                79, 86, 71, 68, 84, 90, 76, 63, 87, 80, 73, 91, 65, 82, 75];

console.log("人数:", scores.length);
console.log("平均:", round(mean(scores)));
console.log("中央値:", median(scores));
console.log("最頻値:", mode(scores));
console.log("最小:", Math.min(...scores), " 最大:", Math.max(...scores));

**結果の読み方**: 平均と中央値が近い（約 78）ので、分布は左右対称に近いと考えられます。
極端に大きい値（外れ値）があると平均は引っ張られますが、中央値はほとんど影響を受けません。年収や住宅価格のように偏った分布では、中央値のほうが「ふつう」を表すのに適しています。

### 例題 2: 外れ値が平均に与える影響

例題 1 のデータに、入力ミスで「950 点」が 1 人追加されたとします。平均と中央値はどう変わるでしょうか。

In [ ]:
const scoresWithTypo = [...scores, 950];

htmlTable(["", "平均", "中央値"], [
  ["元のデータ", mean(scores), median(scores)],
  ["950 を追加", mean(scoresWithTypo), median(scoresWithTypo)],
])

**結果の読み方**: たった 1 つの異常値で平均は約 28 点も上がりましたが、中央値はほとんど変わりません。分析の前に **データの確認（外れ値・入力ミスのチェック）** が重要な理由です。

### 例題 3: 散らばり（分散・標準偏差・範囲・四分位範囲）

2 つのクラスの点数があります。平均はほぼ同じですが、散らばり方が違います。

In [ ]:
const classA = [70, 72, 75, 78, 80, 80, 82, 85, 88, 90];
const classB = [40, 55, 65, 75, 80, 80, 85, 95, 100, 100];

htmlTable(["クラス", "平均", "標準偏差", "範囲", "四分位範囲 (IQR)"], [
  ["A", mean(classA), sd(classA), Math.max(...classA) - Math.min(...classA), iqr(classA)],
  ["B", mean(classB), sd(classB), Math.max(...classB) - Math.min(...classB), iqr(classB)],
])

**結果の読み方**: 平均はどちらも 80 ですが、クラス B は標準偏差が約 3 倍で、点数のばらつきが大きいことが分かります。
標準偏差は「平均からの典型的なずれの大きさ」です。四分位範囲は上位・下位 25% を除いた真ん中 50% の幅で、外れ値の影響を受けにくい散らばりの指標です。

> **標本分散と母分散**: `sd(a)` は n − 1 で割る **標本標準偏差**（手元のデータから母集団を推定するときの標準）です。データ全体そのものの散らばりを知りたいときは `sd(a, 0)`（n で割る）を使います。

### 例題 4: 箱ひげ図で分布を比べる

例題 3 の 2 クラスを箱ひげ図で比較します。箱は第 1 四分位数〜第 3 四分位数、赤い線は中央値、ひげは箱から IQR の 1.5 倍以内の範囲、○ は外れ値です。

In [ ]:
boxPlot({ "クラス A": classA, "クラス B": classB }, { title: "2 クラスの点数の分布", yLabel: "点数" })

### 例題 5: 度数分布表とヒストグラム

例題 1 の点数を 10 点刻みの階級に分け、度数分布表とヒストグラムを作ります。`bins` を変えると印象が変わることも確認しましょう。

In [ ]:
function frequencyTable(data, start, step, count) {
  const rows = [];
  for (let i = 0; i < count; i++) {
    const lo = start + i * step, hi = lo + step;
    const n = data.filter((v) => v >= lo && v < hi).length;
    rows.push([`${lo} 〜 ${hi - 1}`, n, round(n / data.length * 100, 1) + " %", "■".repeat(n)]);
  }
  return htmlTable(["階級", "度数", "相対度数", ""], rows);
}

frequencyTable(scores, 50, 10, 5)

In [ ]:
histogram(scores, { bins: 5, title: "点数のヒストグラム（bins = 5）", xLabel: "点数" })

In [ ]:
histogram(scores, { bins: 15, title: "点数のヒストグラム（bins = 15）", xLabel: "点数" })

**結果の読み方**: 階級の幅（`bins`）が細かすぎると凸凹が目立ち、粗すぎると形が分かりません。いくつか試して、分布の形（山が 1 つか 2 つか、左右対称か）をつかむのがコツです。

### 例題 6: 外れ値の検出（IQR ルールと z スコア）

ある店舗の 1 日の売上（万円）です。外れ値を 2 つの方法で見つけます。

- **IQR ルール**: 第 1 四分位数 − 1.5 × IQR より小さい、または 第 3 四分位数 + 1.5 × IQR より大きい値
- **z スコア**: 平均から標準偏差の何倍離れているか。|z| > 2 や > 3 を外れ値とすることが多い

In [ ]:
const dailySales = [42, 45, 39, 51, 48, 44, 47, 120, 43, 46, 41, 50, 38, 49, 45, 12, 44, 47, 46, 43];

const q1 = quantile(dailySales, 0.25), q3 = quantile(dailySales, 0.75), range = q3 - q1;
const lower = q1 - 1.5 * range, upper = q3 + 1.5 * range;
console.log(`IQR ルールの範囲: ${round(lower)} 〜 ${round(upper)}`);
console.log("IQR ルールで外れ値:", dailySales.filter((v) => v < lower || v > upper));

const z = zscores(dailySales);
const zOutliers = dailySales.filter((v, i) => Math.abs(z[i]) > 2);
console.log("z スコア（|z| > 2）で外れ値:", zOutliers);

htmlTable(["値", "z スコア"], dailySales.map((v, i) => [v, z[i]]).filter((r) => Math.abs(r[1]) > 1))

**結果の読み方**: 120 と 12 はどちらの方法でも外れ値と判定されました。ただし z スコアは、外れ値自身が平均と標準偏差を押し広げるため、外れ値が多いと見つけにくくなります（IQR ルールのほうが頑健です）。
外れ値は「除外する」のではなく、まず **原因を調べる**（入力ミスか、特売日のような本当の出来事か）のが基本です。

### 例題 7: 標準化と偏差値

科目によって平均も散らばりも違うテストの点数は、そのままでは比べられません。**標準化**（z スコア）すると「平均 0、標準偏差 1」にそろえて比較できます。偏差値は z スコアを 50 ± 10 の尺度にしたものです。

数学 65 点（クラス平均 50、SD 15）と英語 75 点（クラス平均 70、SD 5）は、どちらが相対的に優れているでしょうか。

In [ ]:
function deviationScore(x, mu, sigma) {
  return 50 + 10 * (x - mu) / sigma;
}

const mathZ = (65 - 50) / 15;
const engZ = (75 - 70) / 5;
htmlTable(["科目", "点数", "平均", "SD", "z スコア", "偏差値"], [
  ["数学", 65, 50, 15, mathZ, deviationScore(65, 50, 15)],
  ["英語", 75, 70, 5, engZ, deviationScore(75, 70, 5)],
])

**結果の読み方**: 点数は英語のほうが高いのに、偏差値で見ると同じ 60 です。どちらも「平均より標準偏差 1 つ分上」で、相対的な位置は同じということです。

### 例題 8: 分布の形（歪度・尖度）

- **歪度（わいど）**: 分布の左右の偏り。0 で対称、正なら右に長い裾、負なら左に長い裾
- **尖度（せんど）**: 分布の尖り具合・裾の重さ。正規分布で 0（ここでの定義）

対称な正規乱数と、右に裾を引く指数分布風のデータを比べます。

In [ ]:
setSeed(3);
const symmetric = generate(2000, () => randNormal(100, 15));
const skewed = generate(2000, () => -Math.log(1 - rand()) * 30 + 50);   // 指数分布（右に長い裾）

htmlTable(["データ", "平均", "中央値", "歪度", "尖度"], [
  ["対称（正規）", mean(symmetric), median(symmetric), skewness(symmetric), kurtosis(symmetric)],
  ["右に偏り（指数）", mean(skewed), median(skewed), skewness(skewed), kurtosis(skewed)],
])

In [ ]:
histogram(skewed, { bins: 30, title: "右に裾を引く分布（歪度 > 0）: 平均 > 中央値", xLabel: "値" })

**結果の読み方**: 右に裾を引く分布では、少数の大きい値に引っ張られて **平均 > 中央値** になります。歪度が大きいデータに平均や標準偏差だけを使うと実態を見誤ることがあるため、中央値や四分位数も併記しましょう。

### 例題 9: グループ別集計とクロス集計

「オブジェクトの配列」形式のデータ（表形式データ）を、グループごとに集計します。pandas の `groupby` や `crosstab` にあたる処理を自分で書きます。

In [ ]:
const orders = [
  { region: "東京", category: "食品", amount: 3200 }, { region: "東京", category: "衣料", amount: 8900 },
  { region: "大阪", category: "食品", amount: 2800 }, { region: "東京", category: "食品", amount: 4100 },
  { region: "大阪", category: "衣料", amount: 12000 }, { region: "名古屋", category: "食品", amount: 3500 },
  { region: "名古屋", category: "衣料", amount: 7600 }, { region: "大阪", category: "食品", amount: 3900 },
  { region: "東京", category: "衣料", amount: 15000 }, { region: "名古屋", category: "食品", amount: 2900 },
  { region: "大阪", category: "衣料", amount: 9800 }, { region: "東京", category: "食品", amount: 3700 },
];

// グループ化: キーごとに行をまとめる
function groupBy(rows, keyFn) {
  const groups = new Map();
  for (const r of rows) {
    const k = keyFn(r);
    if (!groups.has(k)) groups.set(k, []);
    groups.get(k).push(r);
  }
  return groups;
}

const byRegion = groupBy(orders, (r) => r.region);
const regionRows = [...byRegion].map(([region, rows]) => {
  const amounts = rows.map((r) => r.amount);
  return [region, rows.length, sum(amounts), mean(amounts), Math.max(...amounts)];
});
htmlTable(["地域", "件数", "合計", "平均", "最大"], regionRows)

In [ ]:
// クロス集計: 地域 × カテゴリの合計金額
const regions = [...new Set(orders.map((r) => r.region))];
const categories = [...new Set(orders.map((r) => r.category))];
const crossRows = regions.map((region) => [
  region,
  ...categories.map((c) => sum(orders.filter((r) => r.region === region && r.category === c).map((r) => r.amount))),
]);
htmlTable(["地域 \\ カテゴリ", ...categories], crossRows)

In [ ]:
barChart(regionRows.map((r) => r[0]), regionRows.map((r) => r[2]), { title: "地域別の売上合計", yLabel: "金額（円）" })

## 3. 確率分布とシミュレーション

「もしランダムだったら、どのくらいの確率でこの結果になるか」を考える道具が確率分布です。仮説検定の土台になります。

### 例題 10: 二項分布 — コインを 10 回投げて表が 8 回以上出る確率

公正なコイン（表の確率 0.5）を 10 回投げるとき、表の回数は **二項分布** に従います。表が 8 回以上出る確率を求め、分布をグラフにします。

In [ ]:
const n = 10, p = 0.5;
const ks = [...Array(n + 1).keys()];                       // 0, 1, ..., 10
const probs = ks.map((k) => binomialPmf(k, n, p));

const pAtLeast8 = sum(ks.filter((k) => k >= 8).map((k) => binomialPmf(k, n, p)));
console.log("表が 8 回以上出る確率:", round(pAtLeast8, 4), `(約 ${round(pAtLeast8 * 100, 1)} %)`);
console.log("期待値 n × p =", n * p, " 標準偏差 √(np(1-p)) =", round(Math.sqrt(n * p * (1 - p)), 3));

barChart(ks.map(String), probs, { title: "二項分布 B(10, 0.5)", xLabel: "表の回数", yLabel: "確率" })

**結果の読み方**: 8 回以上表が出る確率は約 5.5%。「20 回に 1 回くらいは起こる」ので、それだけでコインが歪んでいるとは言い切れません。この「偶然でも起こる確率」の考え方が、後の p 値につながります。

### 例題 11: ポアソン分布 — 1 時間に平均 4 人来る店に 8 人以上来る確率

「一定時間に起こる回数」はポアソン分布でモデル化できます。平均 λ = 4 のとき、8 人以上来る確率はどのくらいでしょうか。

In [ ]:
const lambda = 4;
const counts = [...Array(15).keys()];
const pmf = counts.map((k) => poissonPmf(k, lambda));

const pAtLeast8 = 1 - sum(counts.filter((k) => k <= 7).map((k) => poissonPmf(k, lambda)));
console.log("8 人以上来る確率:", round(pAtLeast8, 4));
console.log("ちょうど 4 人の確率:", round(poissonPmf(4, lambda), 4));

barChart(counts.map(String), pmf, { title: "ポアソン分布 (λ = 4)", xLabel: "1 時間の来客数", yLabel: "確率" })

**結果の読み方**: 8 人以上は約 5%。ポアソン分布は「平均 = 分散 = λ」という性質があり、待ち行列や故障回数、アクセス数などの分析に使われます。

### 例題 12: 正規分布 — 身長 180 cm 以上の割合と、上位 5% の境界

成人男性の身長が平均 171 cm、標準偏差 6 cm の正規分布に従うとします。

1. 180 cm 以上の人の割合
2. 165 cm 以上 175 cm 以下の人の割合
3. 上位 5% に入る身長の境界

In [ ]:
const mu = 171, sigma = 6;

console.log("180 cm 以上の割合:", round(1 - normalCdf(180, mu, sigma), 4));
console.log("165〜175 cm の割合:", round(normalCdf(175, mu, sigma) - normalCdf(165, mu, sigma), 4));
console.log("上位 5% の境界:", round(mu + sigma * normalQuantile(0.95), 1), "cm");
console.log("平均 ± 1SD に入る割合:", round(normalCdf(1) - normalCdf(-1), 4), "（約 68%）");
console.log("平均 ± 2SD に入る割合:", round(normalCdf(2) - normalCdf(-2), 4), "（約 95%）");

const xs = generate(121, (_, i) => 150 + i * 0.35);
lineChart([{ name: "確率密度", values: xs.map((x) => normalPdf(x, mu, sigma)) }],
  { xValues: xs, title: "正規分布 N(171, 6²)", xLabel: "身長 (cm)", yLabel: "密度" })

**結果の読み方**: 180 cm 以上は約 6.7%、上位 5% の境界は約 180.9 cm です。「平均 ± 1SD に約 68%、± 2SD に約 95%」は正規分布の重要な性質で、標準偏差の意味を直感的につかむのに役立ちます。

### 例題 13: 大数の法則 — 試行回数を増やすと平均は真の値に近づく

サイコロを振る回数を増やしながら、出た目の平均を記録します。平均は理論値 3.5 に近づいていくでしょうか。

In [ ]:
setSeed(10);
let total = 0;
const runningMean = [];
for (let i = 1; i <= 2000; i++) {
  total += randInt(1, 6);
  runningMean.push(total / i);
}

console.log("10 回後の平均:", round(runningMean[9], 3));
console.log("100 回後の平均:", round(runningMean[99], 3));
console.log("2000 回後の平均:", round(runningMean[1999], 3));

lineChart([{ name: "出た目の平均", values: runningMean }, { name: "理論値 3.5", values: runningMean.map(() => 3.5) }],
  { title: "大数の法則: サイコロの平均", xLabel: "試行回数", yLabel: "平均" })

### 例題 14: 中心極限定理 — 「平均の分布」は正規分布に近づく

サイコロ 1 個の目の分布は一様（どの目も 1/6）ですが、**サイコロ 30 個の目の平均** を何度も取ると、その分布は正規分布に近づきます。これが中心極限定理で、t 検定などが正規分布を前提にできる理由です。

In [ ]:
setSeed(11);
const singleDie = generate(5000, () => randInt(1, 6));
const meanOf30 = generate(5000, () => mean(generate(30, () => randInt(1, 6))));

console.log("サイコロ 1 個: 平均", round(mean(singleDie), 3), " SD", round(sd(singleDie), 3));
console.log("30 個の平均:  平均", round(mean(meanOf30), 3), " SD", round(sd(meanOf30), 3),
            "（理論値: SD = 1.708 / √30 =", round(1.708 / Math.sqrt(30), 3) + "）");

histogram(singleDie, { bins: 6, title: "サイコロ 1 個の目（一様分布）", xLabel: "目" })

In [ ]:
histogram(meanOf30, { bins: 30, title: "サイコロ 30 個の平均の分布（正規分布に近い）", xLabel: "平均" })

**結果の読み方**: 平均の散らばり（**標準誤差**）は元の SD を √n で割った値になります。n を大きくするほど平均の推定は正確になりますが、精度は √n でしか上がりません（4 倍のデータで精度 2 倍）。

### 例題 15: 確率分布と実データの比較（正規性の目視チェック）

正規分布を仮定する手法（t 検定など）を使う前に、データが正規分布に近いか確認します。ここではヒストグラムと、理論的な正規分布の密度を比べます。

In [ ]:
setSeed(12);
const weights = generate(300, () => randNormal(500, 8));   // 製品の重さ（g）の測定値

const s = summary(weights);
const inside1sd = weights.filter((w) => Math.abs(w - s.mean) < s.sd).length / weights.length;
const inside2sd = weights.filter((w) => Math.abs(w - s.mean) < 2 * s.sd).length / weights.length;
console.log("平均 ± 1SD に入る割合:", round(inside1sd, 3), "（正規分布なら約 0.683）");
console.log("平均 ± 2SD に入る割合:", round(inside2sd, 3), "（正規分布なら約 0.954）");
console.log("歪度:", round(skewness(weights), 3), " 尖度:", round(kurtosis(weights), 3), "（正規分布なら 0 に近い）");

histogram(weights, { bins: 20, title: "製品の重さの分布", xLabel: "重さ (g)" })

## 4. 推定

手元の標本から、母集団の値（母平均・母比率）を **幅を持って** 推定するのが区間推定です。

### 例題 16: 母平均の 95% 信頼区間

工場で作った製品 20 個の重さを測りました。全製品（母集団）の平均の重さを 95% 信頼区間で推定します。

$$\bar{x} \pm t_{0.975,\,n-1} \times \frac{s}{\sqrt{n}}$$

In [ ]:
function ciMean(a, conf = 0.95) {
  const n = a.length, m = mean(a), se = sd(a) / Math.sqrt(n);
  const t = tQuantile(1 - (1 - conf) / 2, n - 1);
  return { mean: m, se, lower: m - t * se, upper: m + t * se, t };
}

const sample20 = [498.2, 501.5, 499.8, 502.1, 497.6, 500.9, 503.2, 499.1, 498.7, 501.8,
                  500.3, 497.9, 502.6, 499.5, 500.7, 498.4, 501.2, 499.9, 502.8, 500.1];
const ci = ciMean(sample20);
console.log("標本平均:", round(ci.mean, 2), " 標準誤差:", round(ci.se, 3), " t(0.975, 19) =", round(ci.t, 3));
console.log(`95% 信頼区間: ${round(ci.lower, 2)} 〜 ${round(ci.upper, 2)}`);

**結果の読み方**: 「母平均は約 499.6〜501.0 g の範囲にある」と推定されます。
95% 信頼区間の正しい意味は「同じ方法で何度も標本を取って区間を作ると、そのうち 95% が母平均を含む」です。次の例題でこれを確かめます。

### 例題 17: 信頼区間の意味をシミュレーションで確かめる

母平均 500 の母集団から n = 20 の標本を 200 回取り、毎回 95% 信頼区間を作ります。母平均 500 を含む区間は何回あるでしょうか。

In [ ]:
setSeed(20);
let contains = 0;
const trials = 200;
const firstFew = [];
for (let i = 0; i < trials; i++) {
  const samp = generate(20, () => randNormal(500, 8));
  const c = ciMean(samp);
  if (c.lower <= 500 && 500 <= c.upper) contains++;
  if (i < 8) firstFew.push([i + 1, c.lower, c.upper, c.lower <= 500 && 500 <= c.upper ? "○" : "×"]);
}
console.log(`母平均を含んだ区間: ${contains} / ${trials} 回（${round(contains / trials * 100, 1)} %）`);
htmlTable(["回", "下限", "上限", "500 を含む"], firstFew)

### 例題 18: 母比率の信頼区間とブートストラップ信頼区間

**(a)** アンケートで 400 人中 128 人が「満足」と答えました。満足率の 95% 信頼区間を求めます（正規近似）。

**(b)** 分布の形が分からない統計量（ここでは **中央値**）の信頼区間は、公式がありません。そこで **ブートストラップ法**（標本から復元抽出を繰り返して統計量の散らばりを調べる）で求めます。

In [ ]:
// (a) 母比率
function ciProportion(x, n, conf = 0.95) {
  const p = x / n, se = Math.sqrt(p * (1 - p) / n);
  const z = normalQuantile(1 - (1 - conf) / 2);
  return { p, se, lower: p - z * se, upper: p + z * se };
}
const cp = ciProportion(128, 400);
console.log(`満足率 ${round(cp.p * 100, 1)} %、95% 信頼区間: ${round(cp.lower * 100, 1)} 〜 ${round(cp.upper * 100, 1)} %`);

// (b) ブートストラップ
function bootstrapCI(a, statFn, iterations = 2000, conf = 0.95) {
  const stats = generate(iterations, () => statFn(resample(a)));
  return { lower: quantile(stats, (1 - conf) / 2), upper: quantile(stats, 1 - (1 - conf) / 2), stats };
}
setSeed(21);
const incomes = [310, 280, 450, 390, 1200, 330, 520, 300, 410, 350, 290, 640, 380, 460, 2100, 340, 420, 360, 500, 390];
const bs = bootstrapCI(incomes, median);
console.log(`年収（万円）の中央値: ${median(incomes)}、ブートストラップ 95% 信頼区間: ${round(bs.lower)} 〜 ${round(bs.upper)}`);

histogram(bs.stats, { bins: 25, title: "ブートストラップで得た中央値の分布（2000 回）", xLabel: "中央値" })

**結果の読み方**: ブートストラップは「母集団の代わりに標本を使って、抽出をやり直す実験をコンピュータで行う」方法です。分布の仮定がいらず、どんな統計量にも使えるのが強みです（ただし標本が母集団をよく代表していることが前提）。

## 5. 仮説検定

### 検定の考え方

1. **帰無仮説 H₀**（「差はない」「効果はない」）を立てる
2. H₀ が正しいと仮定したときに、手元のデータ（またはもっと極端なデータ）が得られる確率 = **p 値** を計算する
3. p 値が **有意水準**（ふつう 0.05）より小さければ、H₀ を棄却し「差がある（有意）」と判断する

| 用語 | 意味 |
|---|---|
| 第 1 種の誤り | 本当は差がないのに「ある」と判断（確率 = 有意水準） |
| 第 2 種の誤り | 本当は差があるのに「ない」と判断 |
| 検出力 | 本当に差があるとき、正しく「ある」と判断できる確率 |

> p 値は「H₀ が正しい確率」ではありません。「H₀ のもとでこのデータが偶然出る確率」です。また、有意 = 重要 ではありません。効果の大きさ（効果量）も必ず見ましょう。

### 例題 19: 1 標本 t 検定 — 製品の平均重量は 500 g と言えるか

例題 16 の 20 個の重さのデータで、「母平均 = 500 g」という帰無仮説を検定します。

In [ ]:
function tTestOneSample(a, mu0) {
  const n = a.length, m = mean(a), se = sd(a) / Math.sqrt(n);
  const t = (m - mu0) / se;
  const df = n - 1;
  const p = 2 * (1 - tCdf(Math.abs(t), df));   // 両側検定
  return { t, df, p, mean: m, se };
}

const res1 = tTestOneSample(sample20, 500);
console.log("標本平均:", round(res1.mean, 3), " t 値:", round(res1.t, 3), " 自由度:", res1.df, " p 値:", round(res1.p, 4));
console.log(res1.p < 0.05 ? "→ 有意水準 5% で母平均は 500 g と異なると言える" : "→ 母平均が 500 g と異なるとは言えない");

**結果の読み方**: p 値が 0.05 より大きいので、「平均は 500 g と異なる」とは言えません。ただし、これは「500 g である」ことの証明ではなく、「500 g でないと言えるほどの証拠はない」という意味です。

### 例題 20: 2 標本 t 検定（Welch） — A/B テストで滞在時間は変わったか

Web サイトの新デザイン（B）は旧デザイン（A）より滞在時間（秒）が長いでしょうか。2 つの独立したグループの平均を比べます。
分散が等しいと仮定しない **Welch の t 検定** を使うのが現代の標準です。

In [ ]:
function tTestWelch(a, b) {
  const na = a.length, nb = b.length, va = variance(a), vb = variance(b);
  const se = Math.sqrt(va / na + vb / nb);
  const t = (mean(a) - mean(b)) / se;
  const df = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1));
  const p = 2 * (1 - tCdf(Math.abs(t), df));
  return { t, df, p, diff: mean(a) - mean(b), se };
}

const stayA = [45, 52, 38, 60, 41, 55, 47, 39, 50, 44, 58, 42, 49, 36, 53, 46, 40, 51, 48, 43];
const stayB = [58, 63, 49, 71, 55, 66, 60, 52, 64, 57, 69, 54, 61, 50, 65, 59, 53, 62, 67, 56];

const res2 = tTestWelch(stayB, stayA);
console.log("A の平均:", round(mean(stayA), 1), " B の平均:", round(mean(stayB), 1), " 差:", round(res2.diff, 2));
console.log("t 値:", round(res2.t, 3), " 自由度:", round(res2.df, 1), " p 値:", res2.p < 0.0001 ? "< 0.0001" : round(res2.p, 4));

boxPlot({ "A（旧）": stayA, "B（新）": stayB }, { title: "滞在時間の比較", yLabel: "秒" })

**結果の読み方**: p 値が非常に小さいので、B の滞在時間は A より有意に長いと言えます。差は約 12 秒です。「有意かどうか」だけでなく、「12 秒の差はビジネス上意味があるか」を考えるのが分析者の仕事です。

### 例題 21: 対応のある t 検定 — トレーニング前後で記録は伸びたか

同じ 12 人の、トレーニング前後の腕立て伏せの回数です。**同じ人の前後** のデータなので、個人差を取り除ける「対応のある t 検定」を使います（差を取って 1 標本 t 検定をするのと同じです）。

In [ ]:
function tTestPaired(before, after) {
  const diffs = after.map((v, i) => v - before[i]);
  const r = tTestOneSample(diffs, 0);
  return { ...r, meanDiff: mean(diffs), diffs };
}

const before = [20, 25, 18, 30, 22, 27, 15, 24, 19, 28, 21, 23];
const after  = [24, 27, 22, 33, 21, 30, 19, 28, 20, 31, 25, 26];

const res3 = tTestPaired(before, after);
console.log("平均の伸び:", round(res3.meanDiff, 2), "回  t 値:", round(res3.t, 3), " p 値:", round(res3.p, 4));
htmlTable(["人", "前", "後", "差"], before.map((b, i) => [i + 1, b, after[i], after[i] - b]))

**結果の読み方**: 平均で約 3 回伸びており、p 値 < 0.05 なので有意な改善です。もし 2 標本 t 検定（対応を無視）で計算すると、個人差の分だけ散らばりが大きく見積もられ、有意になりにくくなります。データの構造に合った検定を選びましょう。

### 例題 22: 母比率の検定 — コンバージョン率は 3% から改善したか

これまでのコンバージョン率は 3% でした。新しいページでは 2000 人中 80 人（4%）が購入しました。改善したと言えるでしょうか（正規近似による z 検定）。
また、2 つのページの比率を直接比べる **2 標本の比率の検定** も行います。

In [ ]:
function proportionTest(x, n, p0) {
  const p = x / n;
  const z = (p - p0) / Math.sqrt(p0 * (1 - p0) / n);
  return { p, z, pValue: 2 * (1 - normalCdf(Math.abs(z))) };
}

function twoProportionTest(x1, n1, x2, n2) {
  const p1 = x1 / n1, p2 = x2 / n2, pooled = (x1 + x2) / (n1 + n2);
  const z = (p1 - p2) / Math.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2));
  return { p1, p2, z, pValue: 2 * (1 - normalCdf(Math.abs(z))) };
}

const r1 = proportionTest(80, 2000, 0.03);
console.log(`(1) 新ページ 4.0 % vs 基準 3 %: z = ${round(r1.z, 3)}, p 値 = ${round(r1.pValue, 4)}`);

const r2 = twoProportionTest(80, 2000, 66, 2200);
console.log(`(2) 新ページ ${round(r2.p1 * 100, 2)} % vs 旧ページ ${round(r2.p2 * 100, 2)} %: z = ${round(r2.z, 3)}, p 値 = ${round(r2.pValue, 4)}`);

**結果の読み方**: (1) 基準値 3% との比較では有意ですが、(2) 同時期の旧ページ（3.0%）と直接比べると p 値は 0.05 を上回り、有意とは言えません。比較対象と標本サイズによって結論が変わることがあります。比率の差の検定では、数千人規模のデータでも 1 ポイントの差を検出するのは簡単ではありません。

### 例題 23: カイ二乗検定（適合度） — このサイコロは公正か

サイコロを 120 回振った結果です。各目が 20 回ずつ出ると期待されますが、実際の度数とのずれは偶然の範囲でしょうか。

$$\chi^2 = \sum \frac{(観測度数 - 期待度数)^2}{期待度数}$$

In [ ]:
function chiSquareGof(observed, expected) {
  const chi2 = sum(observed.map((o, i) => (o - expected[i]) ** 2 / expected[i]));
  const df = observed.length - 1;
  return { chi2, df, p: 1 - chi2Cdf(chi2, df) };
}

const observed = [15, 22, 18, 30, 17, 18];
const expected = observed.map(() => 120 / 6);
const gof = chiSquareGof(observed, expected);
console.log("χ² =", round(gof.chi2, 3), " 自由度:", gof.df, " p 値:", round(gof.p, 4));

barChart(["1", "2", "3", "4", "5", "6"], observed, { title: "サイコロの目の度数（期待値は各 20）", xLabel: "目", yLabel: "回数" })

**結果の読み方**: p 値 > 0.05 なので、「公正でない」とは言えません。4 の目が 30 回と多く見えますが、120 回程度ではこのくらいの偏りは偶然でも起こります。

### 例題 24: カイ二乗検定（独立性） — 性別と商品の好みに関係はあるか

アンケートのクロス集計表です。「性別と好みは無関係（独立）」という帰無仮説を検定します。
期待度数は「行の合計 × 列の合計 ÷ 全体」で求めます。

In [ ]:
function chiSquareIndependence(table) {
  const rowTotals = table.map((r) => sum(r));
  const colTotals = table[0].map((_, j) => sum(table.map((r) => r[j])));
  const total = sum(rowTotals);
  const expectedTable = table.map((r, i) => r.map((_, j) => rowTotals[i] * colTotals[j] / total));
  let chi2 = 0;
  table.forEach((r, i) => r.forEach((o, j) => { chi2 += (o - expectedTable[i][j]) ** 2 / expectedTable[i][j]; }));
  const df = (table.length - 1) * (table[0].length - 1);
  return { chi2, df, p: 1 - chi2Cdf(chi2, df), expectedTable };
}

const survey = [
  [45, 30, 25],   // 男性: 商品 A, B, C
  [25, 40, 35],   // 女性
];
const ind = chiSquareIndependence(survey);
console.log("χ² =", round(ind.chi2, 3), " 自由度:", ind.df, " p 値:", round(ind.p, 4));
htmlTable(["", "A", "B", "C"], [
  ["男性（観測）", ...survey[0]], ["男性（期待）", ...ind.expectedTable[0]],
  ["女性（観測）", ...survey[1]], ["女性（期待）", ...ind.expectedTable[1]],
])

**結果の読み方**: p 値 < 0.05 なので、性別と商品の好みには関連があると言えます（男性は A、女性は B・C を選ぶ傾向）。カイ二乗検定は「関連の有無」を判断するもので、「どのくらい強い関連か」は別途、割合の差などで確認します。

### 例題 25: 一元配置分散分析（ANOVA） — 3 種類の肥料で収穫量に差はあるか

3 つ以上のグループの平均を比べるときは、t 検定を繰り返すのではなく **分散分析** を使います（繰り返すと第 1 種の誤りが増えるため）。
「グループ間のばらつき」が「グループ内のばらつき」に比べてどれだけ大きいかを F 値で表します。

In [ ]:
function anovaOneWay(groups) {
  const all = groups.flat();
  const grand = mean(all);
  const k = groups.length, total = all.length;
  const ssBetween = sum(groups.map((g) => g.length * (mean(g) - grand) ** 2));
  const ssWithin = sum(groups.map((g) => sum(g.map((v) => (v - mean(g)) ** 2))));
  const dfB = k - 1, dfW = total - k;
  const F = (ssBetween / dfB) / (ssWithin / dfW);
  return { F, dfB, dfW, p: 1 - fCdf(F, dfB, dfW), ssBetween, ssWithin };
}

const fertilizer = {
  "肥料 A": [52, 55, 49, 58, 53, 51, 56, 54],
  "肥料 B": [60, 63, 58, 65, 61, 59, 64, 62],
  "肥料 C": [54, 57, 52, 59, 55, 53, 58, 56],
};
const an = anovaOneWay(Object.values(fertilizer));
console.log("各グループの平均:", Object.entries(fertilizer).map(([k, v]) => `${k}: ${round(mean(v), 1)}`).join(", "));
console.log("F 値:", round(an.F, 3), ` 自由度 (${an.dfB}, ${an.dfW})`, " p 値:", an.p < 0.0001 ? "< 0.0001" : round(an.p, 4));

boxPlot(fertilizer, { title: "肥料ごとの収穫量", yLabel: "収穫量 (kg)" })

**結果の読み方**: p 値が非常に小さいので「少なくとも 1 組のグループ間に差がある」と言えます。**どのグループ間に差があるか** は ANOVA では分かりません。次の例題のように、多重比較の補正をした上でペアごとに比べます。

### 例題 26: 多重比較 — どのペアに差があるか（Bonferroni 補正）

3 グループなら 3 通りのペアがあります。それぞれ t 検定を行い、**有意水準を検定の回数で割る**（Bonferroni 補正）ことで、全体の第 1 種の誤りを 5% に抑えます。

In [ ]:
const names = Object.keys(fertilizer);
const pairRows = [];
let numTests = 0;
for (let i = 0; i < names.length; i++) {
  for (let j = i + 1; j < names.length; j++) numTests++;
}
const alphaAdjusted = 0.05 / numTests;
for (let i = 0; i < names.length; i++) {
  for (let j = i + 1; j < names.length; j++) {
    const r = tTestWelch(fertilizer[names[i]], fertilizer[names[j]]);
    pairRows.push([`${names[i]} vs ${names[j]}`, r.diff, r.t, r.p, r.p < alphaAdjusted ? "有意" : "—"]);
  }
}
console.log(`検定の回数: ${numTests}、補正後の有意水準: ${round(alphaAdjusted, 4)}`);
htmlTable(["ペア", "平均の差", "t 値", "p 値", "判定"], pairRows)

**結果の読み方**: A–B、B–C の差は有意ですが、A–C の差は補正後の水準では有意ではありません。Bonferroni 補正は簡単ですが保守的（差を見逃しやすい）なので、実務では Tukey の HSD 法などもよく使われます。

### 例題 27: Mann-Whitney の U 検定 — 正規分布を仮定できないときの 2 群比較

顧客満足度（1〜5 の 5 段階）のように、順序はあるが間隔が等しいとは言えないデータや、外れ値が多いデータでは、平均ではなく **順位** に基づく検定を使います。

In [ ]:
function mannWhitney(a, b) {
  const all = [...a, ...b];
  const ranks = rank(all);
  const n1 = a.length, n2 = b.length, total = n1 + n2;
  const r1 = sum(ranks.slice(0, n1));
  const u1 = r1 - n1 * (n1 + 1) / 2;
  const u2 = n1 * n2 - u1;
  const u = Math.min(u1, u2);
  // 同順位（タイ）が多いときの分散の補正
  const tieCounts = new Map();
  for (const v of all) tieCounts.set(v, (tieCounts.get(v) || 0) + 1);
  const tieTerm = sum([...tieCounts.values()].map((t) => t ** 3 - t));
  const mu = n1 * n2 / 2;
  const sigma = Math.sqrt(n1 * n2 / 12 * ((total + 1) - tieTerm / (total * (total - 1))));
  const z = (u - mu + 0.5) / sigma;              // 連続性の補正つき正規近似（scipy と同じ方法）
  return { u1, u2, z, p: 2 * normalCdf(z), medianA: median(a), medianB: median(b) };
}

const storeX = [4, 5, 3, 4, 5, 4, 5, 3, 4, 5, 4, 4, 5, 3, 4];
const storeY = [3, 2, 4, 3, 2, 3, 4, 2, 3, 3, 2, 4, 3, 2, 3];
const mw = mannWhitney(storeX, storeY);
console.log("中央値: X =", mw.medianA, " Y =", mw.medianB);
console.log("U =", Math.min(mw.u1, mw.u2), " z =", round(mw.z, 3), " p 値:", mw.p < 0.0001 ? "< 0.0001" : round(mw.p, 4));

**結果の読み方**: 店舗 X の満足度は Y より有意に高いと言えます。U 検定は「一方の群の値がもう一方より大きくなりやすいか」を調べる検定で、外れ値に強く、正規分布の仮定がいりません。

### 例題 28: 効果量（Cohen's d） — 差の「大きさ」を測る

p 値はサンプルサイズに強く依存します（n が大きければ小さな差でも有意になる）。差の大きさそのものを標準偏差を単位にして表したのが **Cohen's d** です。目安: 0.2 = 小、0.5 = 中、0.8 = 大。

In [ ]:
function cohensD(a, b) {
  const na = a.length, nb = b.length;
  const pooled = Math.sqrt(((na - 1) * variance(a) + (nb - 1) * variance(b)) / (na + nb - 2));
  return (mean(a) - mean(b)) / pooled;
}

const dAB = cohensD(stayB, stayA);
console.log("A/B テスト（例題 20）の効果量 d =", round(dAB, 2));
console.log("肥料 A vs B の効果量 d =", round(cohensD(fertilizer["肥料 B"], fertilizer["肥料 A"]), 2));
console.log("肥料 A vs C の効果量 d =", round(cohensD(fertilizer["肥料 C"], fertilizer["肥料 A"]), 2));

### 例題 29: 並べ替え検定（permutation test） — 分布を仮定しない検定

「2 群に差がない」なら、ラベル（A か B か）を入れ替えても平均の差は変わらないはずです。ラベルをランダムに入れ替えて差を何千回も計算し、実際の差がその中でどれだけ極端かを見るのが並べ替え検定です。数式なしで p 値が求まります。

In [ ]:
function permutationTest(a, b, iterations = 5000) {
  const observedDiff = mean(a) - mean(b);
  const pooled = [...a, ...b];
  let extreme = 0;
  const diffs = [];
  for (let i = 0; i < iterations; i++) {
    const s = shuffle(pooled);
    const d = mean(s.slice(0, a.length)) - mean(s.slice(a.length));
    diffs.push(d);
    if (Math.abs(d) >= Math.abs(observedDiff)) extreme++;
  }
  return { observedDiff, p: extreme / iterations, diffs };
}

setSeed(29);
const perm = permutationTest(stayB, stayA, 5000);
console.log("実際の差:", round(perm.observedDiff, 2), " 並べ替え検定の p 値:", perm.p < 0.0002 ? "< 0.0002" : round(perm.p, 4));
console.log("（例題 20 の Welch の t 検定の p 値とほぼ同じ結論になります）");

histogram(perm.diffs, { bins: 30, title: "ラベルをシャッフルしたときの平均の差の分布（5000 回）", xLabel: "平均の差" })

**結果の読み方**: シャッフルで得られる差はほとんど −6〜6 の範囲に収まり、実際の差（約 12）はその外側にあります。つまり「偶然でこれほどの差が出ることはまずない」ということです。

### 例題 30: 検出力とサンプルサイズ — 何人集めれば差を検出できるか

本当は平均に 3 点の差がある（SD = 10）とき、n を変えて何回検定すると「有意」になるかをシミュレーションします。有意になった割合が **検出力** です。

In [ ]:
setSeed(30);
function estimatePower(n, diff, sigma, trials = 300) {
  let significant = 0;
  for (let i = 0; i < trials; i++) {
    const a = generate(n, () => randNormal(50, sigma));
    const b = generate(n, () => randNormal(50 + diff, sigma));
    if (tTestWelch(a, b).p < 0.05) significant++;
  }
  return significant / trials;
}

const sizes = [10, 20, 40, 80, 160, 320];
const powers = sizes.map((n) => estimatePower(n, 3, 10));
htmlTable(["各群の n", "検出力（有意になった割合）"], sizes.map((n, i) => [n, powers[i]]))

In [ ]:
lineChart([{ name: "検出力", values: powers }, { name: "目安 0.8", values: powers.map(() => 0.8) }],
  { xValues: sizes, title: "サンプルサイズと検出力（差 3, SD 10）", xLabel: "各群の n", yLabel: "検出力" })

**結果の読み方**: 各群 10 人では、本当に差があっても 10% 程度しか検出できません。検出力 0.8（慣例的な目安）に達するには各群 170 人程度が必要です。**実験の前に** 必要なサンプルサイズを見積もることが重要です。

### 例題 31: 多重検定の罠 — 20 回検定すると偶然 1 回は「有意」になる

差がまったくない 2 群を 20 組作り、それぞれ t 検定します。有意水準 5% なら、平均して 1 組は偶然「有意」になります。

In [ ]:
setSeed(31);
const pValues = generate(20, () => tTestWelch(generate(30, () => randNormal(0, 1)), generate(30, () => randNormal(0, 1))).p);
const falsePositives = pValues.filter((p) => p < 0.05).length;
console.log("差がないのに p < 0.05 になった検定:", falsePositives, "/ 20");
console.log("Bonferroni 補正（0.05 / 20 = 0.0025）後:", pValues.filter((p) => p < 0.0025).length, "/ 20");

barChart(pValues.map((_, i) => String(i + 1)), pValues, { title: "20 回の検定の p 値（すべて差なし）", xLabel: "検定", yLabel: "p 値" })

**結果の読み方**: たくさんの仮説を検定して「有意だったものだけ報告する」のは典型的な誤りです（p ハッキング）。検定の数に応じた補正をするか、仮説を事前に絞りましょう。

## 6. 相関と回帰

### 例題 32: 相関係数と散布図 — 広告費と売上

10 店舗の月間広告費（万円）と売上（万円）です。関係の強さをピアソンの相関係数で表し、相関が 0 でないかを検定します。

$$t = \frac{r\sqrt{n-2}}{\sqrt{1-r^2}}\ (\text{自由度 } n-2)$$

In [ ]:
function corrTest(x, y) {
  const r = corr(x, y), n = x.length;
  const t = r * Math.sqrt(n - 2) / Math.sqrt(1 - r * r);
  return { r, t, df: n - 2, p: 2 * (1 - tCdf(Math.abs(t), n - 2)) };
}

const adCost = [10, 15, 8, 20, 12, 25, 18, 5, 22, 14];
const sales  = [120, 150, 100, 210, 130, 240, 170, 80, 220, 140];

const ct = corrTest(adCost, sales);
console.log("相関係数 r =", round(ct.r, 3), " t =", round(ct.t, 3), " p 値:", ct.p < 0.0001 ? "< 0.0001" : round(ct.p, 4));
console.log("スピアマンの順位相関 =", round(spearman(adCost, sales), 3));

scatter(adCost, sales, { title: "広告費と売上", xLabel: "広告費（万円）", yLabel: "売上（万円）" })

**結果の読み方**: r ≒ 0.98 の非常に強い正の相関です。目安として |r| が 0.7 以上で強い、0.4〜0.7 で中程度、0.2 以下でほぼ無相関とされます。
スピアマンの順位相関は値を順位に置き換えて計算するため、外れ値や非線形な単調関係に強い指標です。

### 例題 33: 単回帰分析 — 広告費から売上を予測する

相関が「関係の強さ」なら、回帰は「関係を式で表す」ものです。最小二乗法で直線 `売上 = 切片 + 傾き × 広告費` を求め、決定係数 R² で当てはまりを評価します。

In [ ]:
const reg = linearRegression(adCost, sales);
console.log(`回帰式: 売上 = ${round(reg.intercept, 2)} + ${round(reg.slope, 2)} × 広告費`);
console.log("決定係数 R² =", round(reg.r2, 3), "（売上の変動の", round(reg.r2 * 100, 1), "% を広告費で説明できる）");
console.log("広告費 30 万円のときの予測売上:", round(reg.predict(30), 1), "万円");

scatter(adCost, sales, { title: "回帰直線", xLabel: "広告費（万円）", yLabel: "売上（万円）", line: reg })

**結果の読み方**: 傾き 8.0 は「広告費が 1 万円増えると売上が約 8 万円増える」ことを表します。ただし、広告費 30 万円はデータの範囲（5〜25）の外なので、この予測（**外挿**）は当てにならない可能性があります。

### 例題 34: 残差プロット — 回帰モデルの妥当性を確認する

回帰の前提（残差がランダムに散らばる）が満たされているかを、残差（実測値 − 予測値）のプロットで確認します。残差にパターン（曲線状の並び、広がりの変化）があるなら、直線モデルは不適切です。

In [ ]:
// 本当は 2 次関数の関係があるデータに直線を当てはめてみる
setSeed(34);
const hours = generate(30, (_, i) => 1 + i * 0.3);                            // 学習時間
const testScore = hours.map((h) => 20 + 12 * h - 0.9 * h * h + randNormal(0, 2));   // 山なりの関係

const regH = linearRegression(hours, testScore);
console.log("直線モデルの R² =", round(regH.r2, 3), "（高く見えるが…）");

scatter(hours, regH.residuals, { title: "残差プロット（曲線状のパターン = 直線モデルは不適切）", xLabel: "学習時間", yLabel: "残差" })

In [ ]:
scatter(hours, testScore, { title: "元のデータと直線の当てはめ", xLabel: "学習時間", yLabel: "成績", line: regH })

**結果の読み方**: R² は高くても、残差が「両端で負・真ん中で正」という曲線状のパターンを示しています。この場合は 2 次の項を加える、変数を変換する、といった対応が必要です。**数値だけでなく必ずグラフを見る** ことが大切です。

### 例題 35: アンスコムの例 — 同じ統計量、まったく違うデータ

統計学者アンスコムが作った 4 つのデータセットは、平均・分散・相関係数・回帰直線がほぼ同じなのに、散布図はまったく違います。「要約統計量だけ見て判断してはいけない」ことを示す有名な例です。

In [ ]:
const anscombe = {
  I:   { x: [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], y: [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68] },
  II:  { x: [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], y: [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74] },
  III: { x: [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], y: [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73] },
  IV:  { x: [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],     y: [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89] },
};

htmlTable(["データ", "x の平均", "y の平均", "y の分散", "相関 r", "傾き", "切片", "R²"],
  Object.entries(anscombe).map(([k, d]) => {
    const r = linearRegression(d.x, d.y);
    return [k, mean(d.x), mean(d.y), variance(d.y), corr(d.x, d.y), r.slope, r.intercept, r.r2];
  }))

In [ ]:
scatter(anscombe.II.x, anscombe.II.y, { title: "アンスコム II: 曲線の関係（直線は不適切）", line: linearRegression(anscombe.II.x, anscombe.II.y) })

In [ ]:
scatter(anscombe.IV.x, anscombe.IV.y, { title: "アンスコム IV: 1 点の外れ値が相関を作っている", line: linearRegression(anscombe.IV.x, anscombe.IV.y) })

**結果の読み方**: 4 つとも r ≒ 0.82、回帰直線 y ≒ 3 + 0.5x ですが、II は曲線、III は外れ値 1 つで傾きがずれ、IV は 1 点だけで相関が生まれています。**分析の前に散布図を描く** のは必須です。

### 例題 36: 相関は因果ではない — 交絡変数

「アイスクリームの売上」と「水難事故の件数」には強い正の相関があります。しかしアイスが事故を起こすわけではなく、両方に影響する **気温**（交絡変数）が原因です。シミュレーションで確かめます。

In [ ]:
setSeed(36);
const temperature = generate(60, () => randNormal(25, 6));                        // 気温
const iceCream = temperature.map((t) => 20 + 3 * t + randNormal(0, 8));           // 気温が高いとアイスが売れる
const accidents = temperature.map((t) => 2 + 0.4 * t + randNormal(0, 2));         // 気温が高いと水辺に行く人が増える

console.log("アイス売上と事故件数の相関:", round(corr(iceCream, accidents), 3));
console.log("気温とアイス売上の相関:", round(corr(temperature, iceCream), 3));
console.log("気温と事故件数の相関:", round(corr(temperature, accidents), 3));

// 気温の影響を取り除く（それぞれ気温で回帰した残差同士の相関 = 偏相関）
const resIce = linearRegression(temperature, iceCream).residuals;
const resAcc = linearRegression(temperature, accidents).residuals;
console.log("気温の影響を除いた後の相関（偏相関）:", round(corr(resIce, resAcc), 3), "← ほぼ 0");

**結果の読み方**: 気温の影響を取り除くと、アイスと事故の相関は消えます。相関を見つけたら「第 3 の変数がないか」「逆の因果ではないか」を必ず疑いましょう。因果を示すには、ランダム化実験（A/B テスト）のような設計が必要です。

### 例題 37: 実データの分析 — iris（アヤメ）データセット

このサイトに含まれる `data/iris.csv`（150 本のアヤメの花びら・がく片の寸法と品種）を読み込んで分析します。CSV の読み込みとデータクリーニングも体験しましょう。

In [ ]:
let iris = [];
try {
  const response = await fetch("../files/data/iris.csv");   // ノートブックと同じサイト内のファイル
  const text = await response.text();
  const lines = text.trim().split("\n");
  const header = lines[0].split(",");
  iris = lines.slice(1).map((line) => {
    const cols = line.split(",");
    const row = {};
    header.forEach((h, i) => { row[h] = i < 4 ? Number(cols[i]) : cols[i].trim(); });
    return row;
  });
  console.log(`${iris.length} 行を読み込みました。列: ${header.join(", ")}`);
} catch (error) {
  console.log("読み込みに失敗しました:", error.message);
}

// データの確認: 品種ごとの件数（入力ミスがないか？）
const speciesCount = new Map();
for (const r of iris) speciesCount.set(r.species, (speciesCount.get(r.species) || 0) + 1);
console.log("品種の内訳:", Object.fromEntries(speciesCount));

`se` という品種が 1 件あります。これは `setosa` の入力ミスと考えられるので、修正してから分析します。**カテゴリの値を確認してから分析する** のは基本動作です。

In [ ]:
for (const r of iris) {
  if (r.species === "se") r.species = "setosa";
}

if (iris.length > 0) {
  const species = [...new Set(iris.map((r) => r.species))];
  const petalByspecies = Object.fromEntries(species.map((s) => [s, iris.filter((r) => r.species === s).map((r) => r.petal_length)]));
  console.log("品種ごとの花びらの長さ（平均）:", Object.fromEntries(species.map((s) => [s, round(mean(petalByspecies[s]))])));
  console.log("花びらの長さと幅の相関（全体）:", round(corr(iris.map((r) => r.petal_length), iris.map((r) => r.petal_width)), 3));
  console.log("品種間の差の検定（ANOVA）: p 値", anovaOneWay(Object.values(petalByspecies)).p < 0.0001 ? "< 0.0001" : "≥ 0.0001");
}

In [ ]:
iris.length > 0
  ? boxPlot(Object.fromEntries([...new Set(iris.map((r) => r.species))].map((s) =>
      [s, iris.filter((r) => r.species === s).map((r) => r.petal_length)])), { title: "品種ごとの花びらの長さ", yLabel: "cm" })
  : "データが読み込めていません"

In [ ]:
iris.length > 0
  ? scatter(iris.map((r) => r.petal_length), iris.map((r) => r.petal_width),
      { title: "花びらの長さと幅（3 品種）", xLabel: "花びらの長さ (cm)", yLabel: "花びらの幅 (cm)",
        line: linearRegression(iris.map((r) => r.petal_length), iris.map((r) => r.petal_width)) })
  : "データが読み込めていません"

**結果の読み方**: 花びらの長さは品種で大きく異なり（setosa が小さく、virginica が大きい）、長さと幅には強い正の相関があります。散布図には 3 つのかたまり（クラスタ）が見えます。

## 7. 時系列の基礎

### 例題 38: 移動平均でトレンドを見る

24 か月分の月次売上には、上昇トレンドと季節変動（夏に多い）とノイズが混ざっています。**移動平均** で短期の変動をならしてトレンドを見ます。

In [ ]:
function movingAverage(a, window) {
  return a.map((_, i) => i < window - 1 ? null : mean(a.slice(i - window + 1, i + 1)));
}

setSeed(38);
const months = [...Array(24).keys()].map((i) => i + 1);
const monthlySales = months.map((m) => 100 + 2 * m + 20 * Math.sin((m - 4) / 12 * 2 * Math.PI) + randNormal(0, 8));
const ma3 = movingAverage(monthlySales, 3);
const ma12 = movingAverage(monthlySales, 12);

lineChart([
  { name: "売上", values: monthlySales },
  { name: "3 か月移動平均", values: ma3.map((v, i) => v ?? monthlySales[i]) },
  { name: "12 か月移動平均", values: ma12.map((v, i) => v ?? monthlySales[i]) },
], { xValues: months, title: "月次売上と移動平均", xLabel: "月", yLabel: "売上" })

**結果の読み方**: 3 か月移動平均はノイズを減らしつつ季節変動を残し、12 か月移動平均は季節変動も消してトレンド（上昇傾向）だけを示します。窓の幅は「見たい変動の周期」に合わせて選びます。

### 例題 39: 前年同月比と季節指数

季節変動があるデータでは「先月より減った」だけでは判断できません。**前年同月比** で比べるか、月ごとの **季節指数**（その月の平均 ÷ 全体の平均）で季節性の大きさを見ます。

In [ ]:
const yoy = months.filter((m) => m > 12).map((m) => [m, monthlySales[m - 1], monthlySales[m - 13],
  round((monthlySales[m - 1] / monthlySales[m - 13] - 1) * 100, 1) + " %"]);
htmlTable(["月", "今年", "前年同月", "前年同月比"], yoy)

In [ ]:
const overall = mean(monthlySales);
const seasonal = [...Array(12).keys()].map((i) => mean([monthlySales[i], monthlySales[i + 12]]) / overall);
console.log("季節指数（1 より大きい月は売上が多い月）:");
barChart([...Array(12).keys()].map((i) => (i + 1) + "月"), seasonal, { title: "季節指数", yLabel: "指数" })

## 8. 練習問題

学んだ手法を使って解いてみましょう。「どの手法を使うべきか」を考えるのがポイントです。解答例はその下にあります。

### 問題 1: 2 つの教え方の比較

同じ試験を受けた 2 クラス（別々の生徒）の点数です。教え方 A と B で平均点に差があるか検定し、効果量も求めてください。

```js
const methodA = [68, 72, 75, 80, 64, 78, 70, 74, 69, 77, 73, 71];
const methodB = [74, 79, 82, 85, 70, 84, 77, 80, 76, 83, 78, 81];
```

In [ ]:
// 問題 1: ここにコードを書いてください
const methodA = [68, 72, 75, 80, 64, 78, 70, 74, 69, 77, 73, 71];
const methodB = [74, 79, 82, 85, 70, 84, 77, 80, 76, 83, 78, 81];

**解答例**: 別々の生徒（独立な 2 群）なので Welch の t 検定。

In [ ]:
const q1res = tTestWelch(methodB, methodA);
console.log("平均: A =", round(mean(methodA), 1), " B =", round(mean(methodB), 1), " 差 =", round(q1res.diff, 2));
console.log("t =", round(q1res.t, 3), " p 値 =", round(q1res.p, 4), " 効果量 d =", round(cohensD(methodB, methodA), 2));
console.log(q1res.p < 0.05 ? "→ B のほうが有意に高い（効果量も大きい）" : "→ 有意な差はない");

### 問題 2: 曜日と来客数の関係

ある店の曜日別の来客数（1 週間の合計）です。「曜日によって来客数に偏りはない」という仮説を検定してください。

```js
const visitors = { 月: 80, 火: 75, 水: 82, 木: 78, 金: 95, 土: 130, 日: 120 };
```

In [ ]:
// 問題 2: ここにコードを書いてください
const visitors = { 月: 80, 火: 75, 水: 82, 木: 78, 金: 95, 土: 130, 日: 120 };

**解答例**: 度数データの偏りなので、カイ二乗適合度検定（期待度数は全曜日均等）。

In [ ]:
const obsV = Object.values(visitors);
const expV = obsV.map(() => sum(obsV) / obsV.length);
const q2res = chiSquareGof(obsV, expV);
console.log("χ² =", round(q2res.chi2, 2), " 自由度:", q2res.df, " p 値:", q2res.p < 0.0001 ? "< 0.0001" : round(q2res.p, 4));
console.log("→ 曜日による偏りは有意（週末に集中）");
barChart(Object.keys(visitors), obsV, { title: "曜日別来客数", yLabel: "人" })

### 問題 3: 気温とアイスコーヒーの売上

12 日分の最高気温（℃）とアイスコーヒーの販売数です。相関係数を求め、回帰式を作り、気温 33 ℃の日の販売数を予測してください。散布図も描きましょう。

```js
const temp = [22, 25, 28, 30, 24, 27, 31, 29, 26, 33, 23, 32];
const iced = [45, 58, 72, 85, 52, 66, 90, 80, 60, 98, 48, 92];
```

In [ ]:
// 問題 3: ここにコードを書いてください
const temp = [22, 25, 28, 30, 24, 27, 31, 29, 26, 33, 23, 32];
const iced = [45, 58, 72, 85, 52, 66, 90, 80, 60, 98, 48, 92];

**解答例**

In [ ]:
const q3reg = linearRegression(temp, iced);
console.log("相関係数 r =", round(corr(temp, iced), 3));
console.log(`回帰式: 販売数 = ${round(q3reg.intercept, 1)} + ${round(q3reg.slope, 2)} × 気温`, " R² =", round(q3reg.r2, 3));
console.log("33 ℃ の予測販売数:", round(q3reg.predict(33), 1), "杯");
scatter(temp, iced, { title: "気温とアイスコーヒー販売数", xLabel: "最高気温 (℃)", yLabel: "販売数", line: q3reg })

### 問題 4: 新薬の効果（対応のあるデータ）

10 人の患者の、服薬前と服薬後の血圧です。血圧が下がったと言えるか検定してください。また、平均の変化量の 95% 信頼区間も求めてください。

```js
const bpBefore = [150, 142, 158, 147, 155, 139, 161, 148, 152, 144];
const bpAfter  = [142, 138, 150, 145, 147, 137, 152, 143, 146, 141];
```

In [ ]:
// 問題 4: ここにコードを書いてください
const bpBefore = [150, 142, 158, 147, 155, 139, 161, 148, 152, 144];
const bpAfter  = [142, 138, 150, 145, 147, 137, 152, 143, 146, 141];

**解答例**: 同じ人の前後なので、対応のある t 検定。

In [ ]:
const q4res = tTestPaired(bpBefore, bpAfter);
const q4ci = ciMean(q4res.diffs);
console.log("平均の変化:", round(q4res.meanDiff, 2), " t =", round(q4res.t, 3), " p 値:", round(q4res.p, 5));
console.log(`変化量の 95% 信頼区間: ${round(q4ci.lower, 2)} 〜 ${round(q4ci.upper, 2)}`);
console.log("→ 服薬後に血圧は有意に低下（区間が 0 を含まない）");

## まとめ — どの手法を使うか

| 知りたいこと | データの形 | 手法 |
|---|---|---|
| 1 つの平均が基準値と違うか | 数値 1 群 | 1 標本 t 検定（例題 19） |
| 2 群の平均に差があるか | 独立した 2 群 | Welch の t 検定（例題 20）、並べ替え検定（例題 29） |
| 前後で変化したか | 同じ対象の前後 | 対応のある t 検定（例題 21） |
| 3 群以上の平均に差があるか | 独立した 3 群以上 | 一元配置分散分析（例題 25）＋ 多重比較（例題 26） |
| 正規分布を仮定できない 2 群 | 順位・外れ値の多いデータ | Mann-Whitney の U 検定（例題 27） |
| 比率が基準と違うか / 2 つの比率の差 | 割合 | 比率の z 検定（例題 22） |
| 度数の偏り / カテゴリ間の関連 | 度数・クロス集計表 | カイ二乗検定（例題 23, 24） |
| 2 変数の関係の強さ | 数値 2 変数 | 相関係数と検定（例題 32） |
| 予測式を作る | 数値 2 変数 | 単回帰（例題 33）＋ 残差の確認（例題 34） |
| 公式のない統計量の信頼区間 | 何でも | ブートストラップ（例題 18） |

分析の流れは「**データを確認する（外れ値・欠損・カテゴリ）→ グラフを描く → 要約統計量 → 適切な検定・推定 → 効果量と実務的な意味を考える**」です。

### 次のステップ

- ライブラリを使う: [simple-statistics](https://simple-statistics.github.io/)（このノートブックの関数の多くが揃っています）、[jStat](https://jstat.github.io/)
- グラフ: [Plotly.js](https://plotly.com/javascript/)、[D3.js](https://d3js.org/)、[Chart.js](https://www.chartjs.org/)
- Python 版: `statistics-python.ipynb` では同じテーマを numpy / pandas / scipy / statsmodels で扱います
- 重回帰、ロジスティック回帰、ベイズ統計、時系列モデル（ARIMA）などへ進みましょう

お疲れさまでした！